## Exploring Delta Lake Transaction Logs

Delta Lake stores its transaction log in a `_delta_log/` directory alongside the data files. Each transaction is recorded as a JSON file (`00000000000000000000.json`, etc.). After every 10 commits, a **checkpoint** Parquet file is written for faster state reconstruction.

This notebook will:
1. Create a sample Delta table
2. Perform **INSERT → UPDATE → DELETE → MERGE** to generate multiple log entries
3. Read and visualize the raw `_delta_log` JSON files
4. Show `DESCRIBE HISTORY` for a high-level view 

In [0]:
import json
from pyspark.sql import functions as F
from delta.tables import DeltaTable

table_name = "delta_log_demo"

# Clean up any previous run
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# --- Version 0: CREATE TABLE AS SELECT (initial data) ---
spark.sql(f"""
    CREATE TABLE {table_name} USING DELTA AS
    SELECT * FROM VALUES
        (1, 'Alice',   28, 'Engineering'),
        (2, 'Bob',     35, 'Marketing'),
        (3, 'Charlie', 42, 'Engineering'),
        (4, 'Diana',   31, 'Sales')
    AS t(id, name, age, department)
""")
print("\u2705 Version 0: Table created with 4 rows")

# --- Version 1: UPDATE ---
spark.sql(f"UPDATE {table_name} SET department = 'Sales', age = 36 WHERE id = 2")
print("\u2705 Version 1: Updated Bob's department and age")

# --- Version 2: DELETE ---
spark.sql(f"DELETE FROM {table_name} WHERE id = 4")
print("\u2705 Version 2: Deleted Diana")

# --- Version 3: INSERT (append new rows) ---
spark.sql(f"""
    INSERT INTO {table_name} VALUES 
    (5, 'Eve', 26, 'Data Science'), 
    (6, 'Frank', 39, 'Engineering')
""")
print("\u2705 Version 3: Appended Eve and Frank")

# --- Version 4: MERGE (upsert) ---
spark.sql(f"""
    MERGE INTO {table_name} AS t
    USING (SELECT 2 AS id, 'Bob' AS name, 36 AS age, 'Executive' AS department
           UNION ALL
           SELECT 7, 'Grace', 29, 'Marketing') AS s
    ON t.id = s.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
print("\u2705 Version 4: Merged (updated Bob, inserted Grace)")

# Get the underlying storage location
detail = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]
table_location = detail["location"]
print(f"\n\U0001f4c2 Table location: {table_location}")

print("\n\U0001f4ca Final table state:")
display(spark.sql(f"SELECT * FROM {table_name} ORDER BY id"))

In [0]:
# DESCRIBE HISTORY shows the high-level transaction log maintained by Delta Lake
history_df = spark.sql("DESCRIBE HISTORY delta_log_demo")
display(history_df.select("version", "timestamp", "operation", "operationParameters", "operationMetrics"))

In [0]:
import json
from textwrap import indent

# Retrieve real history to annotate our simulation
hist = spark.sql("DESCRIBE HISTORY delta_log_demo").orderBy("version").collect()

# ============================================================
# Each Delta version is stored as a JSON file in _delta_log/
# e.g., _delta_log/00000000000000000000.json
# Each file contains one JSON object PER LINE (JSON Lines format)
# Below we reconstruct what those files look like.
# ============================================================

def make_log(version_num, actions, description):
    """Pretty-print a simulated delta log version."""
    fname = f"{version_num:020d}.json"
    print("=" * 90)
    print(f"\U0001f4c4  _delta_log/{fname}   \u2014   {description}")
    print("=" * 90)
    for action in actions:
        # Each line in the file is one JSON action
        print(json.dumps(action, indent=2))
    print()

# ----------------------------------------------------------------
# VERSION 0 — CREATE TABLE AS SELECT
# First commit always includes: protocol + metaData + add + commitInfo
# ----------------------------------------------------------------
make_log(0, [
    {
        "protocol": {
            "minReaderVersion": 3,
            "minWriterVersion": 7,
            "readerFeatures": ["deletionVectors"],
            "writerFeatures": ["appendOnly", "deletionVectors", "invariants"]
        }
    },
    {
        "metaData": {
            "id": "421b60e3-ee08-4b27-b7b3-ac4efcd8d4f0",
            "format": {"provider": "parquet", "options": {}},
            "schemaString": json.dumps({
                "type": "struct",
                "fields": [
                    {"name": "id", "type": "integer", "nullable": True, "metadata": {}},
                    {"name": "name", "type": "string", "nullable": True, "metadata": {}},
                    {"name": "age", "type": "integer", "nullable": True, "metadata": {}},
                    {"name": "department", "type": "string", "nullable": True, "metadata": {}}
                ]
            }),
            "partitionColumns": [],
            "configuration": {
                "delta.enableDeletionVectors": "true",
                "delta.parquet.compression.codec": "zstd"
            },
            "createdTime": 1711108085621
        }
    },
    {
        "add": {
            "path": "part-00000-a1b2c3d4-e5f6-7890-abcd-ef1234567890.c000.zstd.parquet",
            "partitionValues": {},
            "size": 1412,
            "modificationTime": 1711108085000,
            "dataChange": True,
            "stats": '{"numRecords":4,"minValues":{"id":1,"name":"Alice","age":28},"maxValues":{"id":4,"name":"Diana","age":42},"nullCount":{"id":0,"name":0,"age":0,"department":0}}'
        }
    },
    {
        "commitInfo": {
            "timestamp": 1711108085621,
            "operation": "CREATE TABLE AS SELECT",
            "operationParameters": {"isManaged": "true", "partitionBy": "[]"},
            "operationMetrics": {"numFiles": "1", "numOutputRows": "4", "numOutputBytes": "1412"},
            "engineInfo": "Databricks-Runtime/<version>",
            "txnId": "abc123-def456-ghi789"
        }
    }
], "CREATE TABLE AS SELECT \u2014 initial 4 rows")

# ----------------------------------------------------------------
# VERSION 1 — UPDATE (with Deletion Vectors)
# Modifies existing data. With DVs, marks rows as deleted in the
# existing file rather than rewriting the whole file.
# ----------------------------------------------------------------
make_log(1, [
    {
        "remove": {
            "path": "part-00000-a1b2c3d4-e5f6-7890-abcd-ef1234567890.c000.zstd.parquet",
            "deletionTimestamp": 1711108090000,
            "dataChange": True,
            "extendedFileMetadata": True,
            "partitionValues": {},
            "size": 1412,
            "deletionVector": {
                "storageType": "u",
                "pathOrInlineDv": "<base85-encoded-dv>",
                "offset": 1,
                "sizeInBytes": 34,
                "cardinality": 1,
                "__comment": "DV marks 1 row (Bob) as deleted in the old file"
            }
        }
    },
    {
        "add": {
            "path": "part-00001-b2c3d4e5-f678-9012-bcde-f12345678901.c000.zstd.parquet",
            "partitionValues": {},
            "size": 978,
            "modificationTime": 1711108090000,
            "dataChange": True,
            "stats": '{"numRecords":1,"minValues":{"id":2,"name":"Bob","age":36},"maxValues":{"id":2,"name":"Bob","age":36}}'
        }
    },
    {
        "commitInfo": {
            "timestamp": 1711108090000,
            "operation": "UPDATE",
            "operationParameters": {"predicate": "(id = 2)"},
            "operationMetrics": {
                "numRemovedFiles": "0", "numAddedFiles": "1",
                "numUpdatedRows": "1", "numCopiedRows": "0",
                "numDeletionVectorsAdded": "1", "numDeletionVectorsRemoved": "0"
            },
            "txnId": "update-txn-001"
        }
    }
], "UPDATE \u2014 Bob's dept changed to Sales, age to 36")

# ----------------------------------------------------------------
# VERSION 2 — DELETE
# Removes rows matching predicate. With DVs, existing files get
# deletion vectors rather than full rewrites.
# ----------------------------------------------------------------
make_log(2, [
    {
        "remove": {
            "path": "part-00000-a1b2c3d4-e5f6-7890-abcd-ef1234567890.c000.zstd.parquet",
            "deletionTimestamp": 1711108095000,
            "dataChange": True,
            "deletionVector": {
                "storageType": "u",
                "pathOrInlineDv": "<base85-encoded-dv>",
                "offset": 1,
                "sizeInBytes": 40,
                "cardinality": 2,
                "__comment": "DV now marks 2 rows as deleted (Bob + Diana)"
            }
        }
    },
    {
        "commitInfo": {
            "timestamp": 1711108095000,
            "operation": "DELETE",
            "operationParameters": {"predicate": "(id = 4)"},
            "operationMetrics": {
                "numRemovedFiles": "0", "numRemovedRows": "1",
                "numCopiedRows": "0", "numAddedFiles": "0",
                "numDeletionVectorsAdded": "1", "numDeletionVectorsRemoved": "1"
            },
            "txnId": "delete-txn-001"
        }
    }
], "DELETE \u2014 removed Diana (id=4)")

# ----------------------------------------------------------------
# VERSION 3 — INSERT (WRITE in Append mode)
# Only adds new files, no removes.
# ----------------------------------------------------------------
make_log(3, [
    {
        "add": {
            "path": "part-00002-c3d4e5f6-7890-1234-cdef-234567890abc.c000.zstd.parquet",
            "partitionValues": {},
            "size": 1050,
            "modificationTime": 1711108100000,
            "dataChange": True,
            "stats": '{"numRecords":2,"minValues":{"id":5,"name":"Eve","age":26},"maxValues":{"id":6,"name":"Frank","age":39}}'
        }
    },
    {
        "commitInfo": {
            "timestamp": 1711108100000,
            "operation": "WRITE",
            "operationParameters": {"mode": "Append"},
            "operationMetrics": {"numFiles": "1", "numOutputRows": "2", "numOutputBytes": "1050"},
            "txnId": "insert-txn-001"
        }
    }
], "INSERT \u2014 appended Eve and Frank")

# ----------------------------------------------------------------
# VERSION 4 — MERGE (upsert)
# Most complex: removes old files for matched rows, adds new files
# for both matched updates and unmatched inserts.
# ----------------------------------------------------------------
make_log(4, [
    {
        "remove": {
            "path": "part-00001-b2c3d4e5-f678-9012-bcde-f12345678901.c000.zstd.parquet",
            "deletionTimestamp": 1711108105000,
            "dataChange": True,
            "size": 978
        }
    },
    {
        "add": {
            "path": "part-00003-d4e5f678-9012-3456-defg-34567890abcd.c000.zstd.parquet",
            "partitionValues": {},
            "size": 1380,
            "modificationTime": 1711108105000,
            "dataChange": True,
            "stats": '{"numRecords":2,"minValues":{"id":2,"name":"Bob","age":29},"maxValues":{"id":7,"name":"Grace","age":36}}'
        }
    },
    {
        "commitInfo": {
            "timestamp": 1711108105000,
            "operation": "MERGE",
            "operationParameters": {
                "predicate": "(t.id = s.id)",
                "matchedPredicates": "[{\"actionType\":\"update\"}]",
                "notMatchedPredicates": "[{\"actionType\":\"insert\"}]"
            },
            "operationMetrics": {
                "numTargetRowsUpdated": "1", "numTargetRowsInserted": "1",
                "numTargetRowsDeleted": "0", "numOutputRows": "2",
                "numSourceRows": "2", "numTargetFilesAdded": "1", "numTargetFilesRemoved": "1"
            },
            "txnId": "merge-txn-001"
        }
    }
], "MERGE \u2014 updated Bob to Executive, inserted Grace")

print("\n" + "=" * 90)
print("\U0001f4a1 KEY OBSERVATIONS:")
print("=" * 90)
print("""
\u2022 Version 0 (CTAS): Contains protocol + metaData + add + commitInfo
  \u2192 protocol & metaData only appear in the FIRST commit (or when schema changes)

\u2022 Version 1 (UPDATE): remove (old data) + add (new data) + commitInfo  
  \u2192 With Deletion Vectors: marks rows deleted in-place instead of rewriting files

\u2022 Version 2 (DELETE): remove + commitInfo
  \u2192 Only removes data, no new files added (with DVs)

\u2022 Version 3 (INSERT): add + commitInfo
  \u2192 Simplest operation \u2014 only adds new Parquet files

\u2022 Version 4 (MERGE): remove + add + commitInfo
  \u2192 Most complex: handles both updates (remove+add) and inserts (add)

\u2022 Every version ends with commitInfo containing operation, parameters, and metrics
\u2022 The 'stats' field in 'add' actions contains min/max/null statistics for query optimization
\u2022 After 10 commits, Delta writes a checkpoint .parquet file for faster state reconstruction
""")

In [0]:
# Visual summary: what action types appear in each version
from pyspark.sql import Row

log_actions = [
    Row(version=0, action="protocol",  description="Sets reader/writer version and features"),
    Row(version=0, action="metaData",  description="Table schema, partition columns, config"),
    Row(version=0, action="add",       description="1 Parquet file (4 rows)"),
    Row(version=0, action="commitInfo", description="CREATE TABLE AS SELECT"),
    Row(version=1, action="remove",    description="Old file (with Deletion Vector)"),
    Row(version=1, action="add",       description="1 new file (updated Bob)"),
    Row(version=1, action="commitInfo", description="UPDATE where id=2"),
    Row(version=2, action="remove",    description="File with expanded DV (Diana deleted)"),
    Row(version=2, action="commitInfo", description="DELETE where id=4"),
    Row(version=3, action="add",       description="1 new file (Eve + Frank)"),
    Row(version=3, action="commitInfo", description="WRITE (Append)"),
    Row(version=4, action="remove",    description="Old Bob file removed"),
    Row(version=4, action="add",       description="1 new file (Bob updated + Grace inserted)"),
    Row(version=4, action="commitInfo", description="MERGE"),
]

summary_df = spark.createDataFrame(log_actions)
display(summary_df.orderBy("version", "action"))

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- DESCRIBE HISTORY shows Delta's actual transaction history
-- Compare this with the raw JSON log structure shown above
DESCRIBE HISTORY delta_log_demo

In [0]:
# Uncomment to clean up
# spark.sql("DROP TABLE IF EXISTS delta_log_demo")
# print("\U0001f9f9 Cleaned up")